In [ ]:
import numpy as np
import pandas as pd
import dai


def main(datasources, start_date, end_date):
    bar1m = datasources["bar1m"]

    sql = f"""
    WITH minute_base AS (
        SELECT
            date::DATE::DATETIME AS date,
            instrument,
            date::TIME AS bar_time,
            deal_number,
            volume
        FROM {bar1m}
    ),
    daily_flow AS (
        SELECT
            date,
            instrument,

            SUM(deal_number) AS deal_number_sum,
            SUM(volume) AS volume_sum,

            SUM(
                CASE
                    WHEN bar_time <= '10:00:00'::TIME
                    THEN deal_number
                    ELSE 0
                END
            ) AS morning_deal_number_sum,

            SUM(
                CASE
                    WHEN bar_time <= '10:00:00'::TIME
                    THEN volume
                    ELSE 0
                END
            ) AS morning_volume_sum

        FROM minute_base
        GROUP BY date, instrument
    ),
    daily_factor AS (
        SELECT
            date,
            instrument,
            (
                morning_deal_number_sum / NULLIF(deal_number_sum, 0)
                -
                morning_volume_sum / NULLIF(volume_sum, 0)
            ) AS factor
        FROM daily_flow
        WHERE deal_number_sum > 0
          AND volume_sum > 0
    )
    SELECT
        date,
        instrument,
        factor
    FROM daily_factor
    ORDER BY date, instrument
    """

    result = dai.query(
        sql,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()

    result = result[["date", "instrument", "factor"]].copy()

    result["date"] = pd.to_datetime(result["date"], errors="coerce")
    result["instrument"] = result["instrument"].astype(str)
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce")

    result = result.replace([np.inf, -np.inf], np.nan)
    result = result.dropna(subset=["date", "instrument", "factor"])
    result = result.drop_duplicates(["date", "instrument"], keep="last")
    result = result.sort_values(["date", "instrument"]).reset_index(drop=True)

    return result[["date", "instrument", "factor"]]